# Trading Card Grader — Full Training Pipeline
---

**Goal:** Train a multi-output CNN to predict card grades (centering, corners, edges, surface)
and map them to PSA / Beckett / SGC / CGC grading scales.

**Pipeline:**
1. Setup & GPU Check
2. Data Collection (eBay scraper or manual dataset)
3. Preprocessing — Validation, Deduplication, Outlier Removal, Augmentation
4. Model Building — Custom CNN, MobileNetV2, EfficientNetB3
5. Training with Callbacks
6. Evaluation & Grade Mapping
7. ONNX Export (for GitHub Pages browser inference)
---

## Step 0: Setup — Install & Import Libraries

In [ ]:
# Install required packages (run once)
!pip install tensorflow tf2onnx onnx scikit-learn pillow matplotlib numpy requests beautifulsoup4 -q
!pip install kagglehub -q  # optional, only needed if using Kaggle datasets

In [ ]:
import os
import sys
import math
import random
import hashlib
import json
import shutil
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2, EfficientNetB3
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from PIL import Image, UnidentifiedImageError

# Reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

print("TensorFlow version:", tf.__version__)
print("All libraries imported successfully!")

In [ ]:
# GPU CHECK
# IMPORTANT: If this prints [] you are on CPU — training will be slow.
# In Google Colab: Runtime → Change runtime type → T4 GPU → Save
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"✅ GPU detected: {gpus}")
    print("   Training will be fast (~30-60s/epoch)")
else:
    print("⚠️  No GPU detected — running on CPU (~5-10 min/epoch)")
    print("   Recommended: Runtime → Change runtime type → T4 GPU")

## Step 1: Configuration

**Image size:** 224×224 — larger than plant project (128×128) because card details
(corner wear, surface scratches) require higher resolution to detect.

**Dataset structure expected:**
```
data/
  train/
    card_001.jpg   ← image file
    card_001.json  ← label: {"centering": 9.0, "corners": 9.5, "edges": 9.0, "surface": 8.5}
    card_002.jpg
    card_002.json
    ...
  val/
    ...
  test/
    ...
```

In [ ]:
# CONFIGURATION
IMG_SIZE       = (224, 224)   # Higher res than plant project — card details need it
BATCH_SIZE     = 32
EPOCHS         = 30           # EarlyStopping will stop before this if val_loss plateaus
LEARNING_RATE  = 1e-3

# Paths — update these to match your dataset location
DATA_DIR       = '../data'
TRAIN_DIR      = os.path.join(DATA_DIR, 'train')
VAL_DIR        = os.path.join(DATA_DIR, 'val')
TEST_DIR       = os.path.join(DATA_DIR, 'test')
MODEL_SAVE_DIR = '../model/saved'

os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

# Grading dimensions — these are the 4 sub-scores models like PSA/BGS use
GRADE_DIMENSIONS = ['centering', 'corners', 'edges', 'surface']

# Weight of each dimension in the final composite score
# Surface & corners are weighted heavier (PSA/BGS emphasis)
GRADE_WEIGHTS = {
    'centering': 0.20,
    'corners':   0.25,
    'edges':     0.25,
    'surface':   0.30
}

print(f"Image size     : {IMG_SIZE[0]} × {IMG_SIZE[1]} pixels")
print(f"Batch size     : {BATCH_SIZE}")
print(f"Max epochs     : {EPOCHS}")
print(f"Grade dims     : {GRADE_DIMENSIONS}")
print(f"Grade weights  : {GRADE_WEIGHTS}")

## Step 2: Data Collection

Two options:
- **Option A** — Use the included eBay scraper (`../data/scraper/ebay_scraper.py`) to auto-collect labeled images
- **Option B** — Manually build a dataset (photos of your own cards + grades you know/submit)

The scraper searches eBay sold listings for graded cards (e.g. "PSA 10 Charizard") and downloads the slab photo + grade label automatically.

**Minimum recommended dataset size:** 500 cards per grade tier (1,000+ total images)

In [ ]:
# OPTION A: Run the eBay scraper
# Uncomment and run this cell to collect data automatically
# See ../data/scraper/ebay_scraper.py for configuration options

# %run ../data/scraper/ebay_scraper.py

# OPTION B: Check your manually built dataset
def check_dataset_structure(data_dir):
    """Verify dataset is structured correctly with matching image + JSON label files."""
    if not os.path.exists(data_dir):
        print(f"⚠️  Directory not found: {data_dir}")
        print("   Run the scraper or manually place images + label JSONs here.")
        return 0

    images = [f for f in os.listdir(data_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    labels = [f for f in os.listdir(data_dir) if f.endswith('.json')]

    # Check each image has a matching label
    matched = 0
    unmatched = []
    for img_file in images:
        base = os.path.splitext(img_file)[0]
        if f"{base}.json" in labels:
            matched += 1
        else:
            unmatched.append(img_file)

    print(f"\n--- Dataset Check: {os.path.basename(data_dir)} ---")
    print(f"  Images found   : {len(images)}")
    print(f"  Labels found   : {len(labels)}")
    print(f"  Matched pairs  : {matched}")
    if unmatched:
        print(f"  ⚠️  Missing labels for: {unmatched[:5]}{'...' if len(unmatched) > 5 else ''}")
    else:
        print(f"  ✅ All images have labels")
    return matched

train_count = check_dataset_structure(TRAIN_DIR)
val_count   = check_dataset_structure(VAL_DIR)
test_count  = check_dataset_structure(TEST_DIR)

print(f"\nTotal labeled samples: {train_count + val_count + test_count}")

if train_count < 100:
    print("\n⚠️  Less than 100 training samples found.")
    print("   The scraper in ../data/scraper/ebay_scraper.py can help collect more.")
    print("   Training will proceed but accuracy will be limited with small datasets.")

## Step 3: Custom Dataset Loader

Unlike the plant project (which used folder-per-class), card grading needs a custom loader
because labels are continuous scores (7.0, 8.5, 9.0...), not discrete categories.
Each image has a paired JSON file with the 4 sub-scores.

In [ ]:
class CardGradingDataset:
    """
    Custom dataset loader for card grading.
    Loads image + JSON label pairs from a directory.

    Label JSON format:
    {
      "centering": 9.0,   # 1.0 - 10.0
      "corners":   9.5,
      "edges":     9.0,
      "surface":   8.5,
      "psa_grade": 9,     # optional: actual PSA grade if known
      "card_name": "Charizard Base Set"  # optional: metadata
    }
    """

    def __init__(self, data_dir, img_size=IMG_SIZE, augment=False):
        self.data_dir = data_dir
        self.img_size = img_size
        self.augment  = augment
        self.samples  = self._load_samples()

    def _load_samples(self):
        """Scan directory for matched image+JSON pairs."""
        samples = []
        if not os.path.exists(self.data_dir):
            return samples
        for fname in sorted(os.listdir(self.data_dir)):
            if not fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                continue
            base      = os.path.splitext(fname)[0]
            img_path  = os.path.join(self.data_dir, fname)
            json_path = os.path.join(self.data_dir, f"{base}.json")
            if not os.path.exists(json_path):
                continue
            with open(json_path) as f:
                label = json.load(f)
            # Validate all required fields exist
            if all(k in label for k in GRADE_DIMENSIONS):
                samples.append({'image': img_path, 'label': label})
        return samples

    def __len__(self):
        return len(self.samples)

    def load_image(self, path):
        """Load and preprocess a single image."""
        img = load_img(path, target_size=self.img_size)
        arr = img_to_array(img) / 255.0  # Normalize to [0, 1]
        return arr

    def get_label_vector(self, label):
        """Extract the 4 sub-scores as a normalized [0,1] vector."""
        return np.array([
            label['centering'] / 10.0,
            label['corners']   / 10.0,
            label['edges']     / 10.0,
            label['surface']   / 10.0
        ], dtype=np.float32)

    def as_tf_dataset(self, batch_size=BATCH_SIZE, shuffle=True):
        """Convert to a tf.data.Dataset for efficient training."""
        images = []
        labels = []
        for sample in self.samples:
            try:
                img = self.load_image(sample['image'])
                lbl = self.get_label_vector(sample['label'])
                images.append(img)
                labels.append(lbl)
            except Exception as e:
                print(f"  Skipping {sample['image']}: {e}")

        images = np.array(images)
        labels = np.array(labels)

        ds = tf.data.Dataset.from_tensor_slices((images, labels))
        if shuffle:
            ds = ds.shuffle(buffer_size=len(images), seed=RANDOM_SEED)
        ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
        return ds, images, labels


# Load datasets
train_dataset = CardGradingDataset(TRAIN_DIR, augment=True)
val_dataset   = CardGradingDataset(VAL_DIR)
test_dataset  = CardGradingDataset(TEST_DIR)

print(f"Training samples  : {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Test samples      : {len(test_dataset)}")

## Step 4: Preprocessing

Adapted from your plant disease pipeline — same 3 steps:
- **4a** Missing/corrupt image removal
- **4b** Duplicate removal (MD5 hash)
- **4c** Outlier detection (PCA + Isolation Forest)
- **4d** Normalization & augmentation

In [ ]:
# 4a — Remove corrupt/unreadable images (same approach as plant project)
def remove_corrupt_images(data_dir):
    """
    Scan every image. Remove corrupt files that can't be decoded.
    Also removes the paired JSON if the image is deleted.
    """
    removed = 0
    checked = 0
    for fname in sorted(os.listdir(data_dir)):
        if not fname.lower().endswith(('.jpg', '.jpeg', '.png')):
            continue
        fpath = os.path.join(data_dir, fname)
        checked += 1
        try:
            with Image.open(fpath) as img:
                img.verify()
            with Image.open(fpath) as img:
                img.load()
        except Exception as e:
            print(f"  [REMOVED] {fname} — {e}")
            os.remove(fpath)
            # Also remove matching JSON label
            json_path = os.path.splitext(fpath)[0] + '.json'
            if os.path.exists(json_path):
                os.remove(json_path)
            removed += 1

    print(f"\n--- Corrupt Image Check: {os.path.basename(data_dir)} ---")
    print(f"  Images checked  : {checked}")
    print(f"  Corrupt removed : {removed}")
    print(f"  Clean images    : {checked - removed}")

for split_dir in [TRAIN_DIR, VAL_DIR, TEST_DIR]:
    if os.path.exists(split_dir):
        remove_corrupt_images(split_dir)

In [ ]:
# 4b — Remove duplicate images (MD5 hash — identical to plant project approach)
def remove_duplicates(data_dir):
    """
    Hash-based deduplication. Deletes any image whose byte content
    has already been seen. Also removes paired JSON.
    """
    seen  = set()
    dupes = 0
    for fname in sorted(os.listdir(data_dir)):
        if not fname.lower().endswith(('.jpg', '.jpeg', '.png')):
            continue
        fpath = os.path.join(data_dir, fname)
        with open(fpath, 'rb') as f:
            h = hashlib.md5(f.read()).hexdigest()
        if h in seen:
            os.remove(fpath)
            json_path = os.path.splitext(fpath)[0] + '.json'
            if os.path.exists(json_path):
                os.remove(json_path)
            dupes += 1
        else:
            seen.add(h)
    print(f"  Duplicates removed from {os.path.basename(data_dir)}: {dupes}")

for split_dir in [TRAIN_DIR, VAL_DIR]:
    if os.path.exists(split_dir):
        remove_duplicates(split_dir)

In [ ]:
# 4c — Outlier detection (PCA + Isolation Forest — same as plant project)
# Detects blurry/mislabeled/extreme images that would hurt training
OUTLIER_SAMPLE_SIZE = 200  # Max images to sample for outlier detection

def detect_outliers(data_dir, sample_size=OUTLIER_SAMPLE_SIZE, img_size=IMG_SIZE):
    if not os.path.exists(data_dir):
        return

    all_imgs = [f for f in os.listdir(data_dir)
                if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    if len(all_imgs) < 10:
        print(f"  Too few images in {os.path.basename(data_dir)} for outlier detection.")
        return

    sample_files = random.sample(all_imgs, min(sample_size, len(all_imgs)))

    od_images = []
    od_paths  = []
    for fname in sample_files:
        fpath = os.path.join(data_dir, fname)
        try:
            arr = img_to_array(load_img(fpath, target_size=img_size)) / 255.0
            od_images.append(arr.flatten())
            od_paths.append(fpath)
        except Exception:
            continue

    od_images = np.array(od_images)
    print(f"  Outlier detection sample: {od_images.shape}")

    # PCA to reduce dimensions before Isolation Forest
    n_components = min(50, od_images.shape[0] - 1, od_images.shape[1])
    pca          = PCA(n_components=n_components, random_state=RANDOM_SEED)
    od_pca       = pca.fit_transform(od_images)
    print(f"  PCA variance explained : {pca.explained_variance_ratio_.sum():.2%}")

    iforest  = IsolationForest(contamination=0.05, random_state=RANDOM_SEED)
    iforest.fit(od_pca)
    scores   = -iforest.score_samples(od_pca)

    threshold    = scores.mean() + 3 * scores.std()
    outlier_idxs = np.where(scores > threshold)[0]

    print(f"  Outliers detected: {len(outlier_idxs)} ({len(outlier_idxs)/len(scores):.1%})")

    # Show outlier images before deleting
    if len(outlier_idxs) > 0:
        show_n = min(4, len(outlier_idxs))
        fig, axes = plt.subplots(1, show_n, figsize=(12, 3))
        if show_n == 1:
            axes = [axes]
        for i, idx in enumerate(outlier_idxs[:show_n]):
            axes[i].imshow(od_images[idx].reshape(img_size[0], img_size[1], 3))
            axes[i].set_title(f"Score: {scores[idx]:.3f}", fontsize=8)
            axes[i].axis('off')
        plt.suptitle(f'Detected Outliers — {os.path.basename(data_dir)}', fontweight='bold')
        plt.tight_layout()
        plt.show()

        for idx in outlier_idxs:
            fpath = od_paths[idx]
            if os.path.exists(fpath):
                os.remove(fpath)
                json_path = os.path.splitext(fpath)[0] + '.json'
                if os.path.exists(json_path):
                    os.remove(json_path)
        print(f"  {len(outlier_idxs)} outlier images removed.")

detect_outliers(TRAIN_DIR)
detect_outliers(VAL_DIR)

In [ ]:
# 4d — Reload cleaned datasets and build tf.data pipelines
# NOTE: no horizontal_flip — card text/art orientation matters!

# Reload after preprocessing cleanup
train_dataset = CardGradingDataset(TRAIN_DIR, augment=True)
val_dataset   = CardGradingDataset(VAL_DIR)

train_ds, X_train, y_train = train_dataset.as_tf_dataset(batch_size=BATCH_SIZE, shuffle=True)
val_ds,   X_val,   y_val   = val_dataset.as_tf_dataset(batch_size=BATCH_SIZE,   shuffle=False)

# Data augmentation layer (built into model via Keras preprocessing layers)
# This applies augmentation ONLY during training, not inference
data_augmentation = keras.Sequential([
    layers.RandomRotation(0.05),              # Small rotation only — extreme = unnatural
    layers.RandomZoom((-0.05, 0.05)),         # Tight zoom range
    layers.RandomBrightness((-0.15, 0.15)),   # Lighting variation
    layers.RandomContrast(0.1),               # Contrast variation
    # NOTE: NO RandomFlip — card orientation matters
], name='data_augmentation')

print(f"Training samples  : {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Steps per epoch   : {math.ceil(len(train_dataset) / BATCH_SIZE)}")

# Visualize a batch of preprocessed card images
if len(X_train) > 0:
    fig, axes = plt.subplots(2, 4, figsize=(14, 7))
    for i, ax in enumerate(axes.flat):
        if i < len(X_train):
            ax.imshow(X_train[i])
            scores_str = ' | '.join([f"{d[0].upper()}: {y_train[i][j]*10:.1f}"
                                     for j, d in enumerate(GRADE_DIMENSIONS)])
            ax.set_title(scores_str, fontsize=6)
            ax.axis('off')
    plt.suptitle('Sample Training Cards (normalized 224×224)', fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("No images loaded yet — add images to the data/train directory first.")

## Step 5: Model Building

Three models (mirrors your plant project structure):

| Model | Type | Notes |
|-------|------|-------|
| Custom CNN | From scratch | Baseline, fast to train |
| MobileNetV2 | Transfer learning | Lightweight, good for deployment |
| EfficientNetB3 | Transfer learning | Best accuracy, slightly larger |

**Key difference from plant project:** Output is 4 regression values [0,1] (not softmax classification).
Loss is MSE per dimension, not categorical crossentropy.

In [ ]:
# 5a — Custom CNN (from scratch baseline)
def build_custom_cnn(img_size=IMG_SIZE):
    """
    Custom CNN for card grading — built from scratch.

    Architecture:
      Augmentation layer
      Block 1: Conv2D(32)  → BatchNorm → MaxPool
      Block 2: Conv2D(64)  → BatchNorm → MaxPool
      Block 3: Conv2D(128) → BatchNorm → MaxPool
      Block 4: Conv2D(256) → BatchNorm → GlobalAvgPool
      Head   : Dense(512) → Dropout → Dense(4, sigmoid)  ← 4 grade dimensions
    """
    inputs = keras.Input(shape=img_size + (3,), name='card_image')
    x = data_augmentation(inputs)

    # Convolutional blocks
    for filters in [32, 64, 128, 256]:
        x = layers.Conv2D(filters, (3, 3), padding='same')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
        x = layers.MaxPooling2D((2, 2))(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(512, activation='relu')(x)
    x = layers.Dropout(0.5)(x)

    # 4 output neurons — one per grading dimension, sigmoid → [0, 1]
    outputs = layers.Dense(4, activation='sigmoid', name='grade_scores')(x)

    return keras.Model(inputs, outputs, name='Custom_CNN')


custom_cnn = build_custom_cnn()
custom_cnn.summary()

In [ ]:
# 5b — MobileNetV2 Transfer Learning
def build_mobilenetv2(img_size=IMG_SIZE, dropout_rate=0.4):
    """
    MobileNetV2 Transfer Learning for card grading.
    Same structure as plant project, output head changed to regression.
    """
    base = MobileNetV2(
        input_shape=img_size + (3,),
        include_top=False,
        weights='imagenet'
    )
    base.trainable = False  # Freeze ImageNet weights

    inputs  = keras.Input(shape=img_size + (3,), name='card_image')
    x       = data_augmentation(inputs)
    x       = base(x, training=False)
    x       = layers.GlobalAveragePooling2D()(x)
    x       = layers.Dense(256, activation='relu')(x)
    x       = layers.Dropout(dropout_rate)(x)
    outputs = layers.Dense(4, activation='sigmoid', name='grade_scores')(x)

    return keras.Model(inputs, outputs, name='MobileNetV2_CardGrader')


mobilenet_model = build_mobilenetv2()
mobilenet_model.summary()

In [ ]:
# 5c — EfficientNetB3 Transfer Learning (upgraded from ResNet50)
# EfficientNetB3 outperforms ResNet50 on fine-grained visual tasks like card grading
def build_efficientnet(img_size=IMG_SIZE, dropout_rate=0.4):
    """
    EfficientNetB3 Transfer Learning.
    Better than ResNet50 for fine-grained detail detection (scratches, corner wear).
    """
    base = EfficientNetB3(
        input_shape=img_size + (3,),
        include_top=False,
        weights='imagenet'
    )
    base.trainable = False

    inputs  = keras.Input(shape=img_size + (3,), name='card_image')
    x       = data_augmentation(inputs)
    x       = base(x, training=False)
    x       = layers.GlobalAveragePooling2D()(x)
    x       = layers.Dense(512, activation='relu')(x)
    x       = layers.Dropout(dropout_rate)(x)
    x       = layers.Dense(256, activation='relu')(x)
    x       = layers.Dropout(0.3)(x)
    outputs = layers.Dense(4, activation='sigmoid', name='grade_scores')(x)

    return keras.Model(inputs, outputs, name='EfficientNetB3_CardGrader')


efficientnet_model = build_efficientnet()
efficientnet_model.summary()

## Step 6: Training

In [ ]:
# Callbacks (same structure as plant project)
def get_callbacks(model_name):
    return [
        EarlyStopping(
            monitor='val_loss',
            patience=6,
            restore_best_weights=True,
            verbose=1
        ),
        ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=3,
            min_lr=1e-7,
            verbose=1
        ),
        ModelCheckpoint(
            filepath=os.path.join(MODEL_SAVE_DIR, f'best_{model_name}.keras'),
            monitor='val_loss',
            save_best_only=True,
            verbose=0
        )
    ]

# Custom MAE metric scaled back to 0-10 (easier to interpret than 0-1)
def grade_mae(y_true, y_pred):
    return tf.reduce_mean(tf.abs(y_true - y_pred)) * 10.0

print("Callbacks and metrics defined.")

In [ ]:
# Train Custom CNN
if len(train_dataset) > 0:
    custom_cnn.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss='mse',
        metrics=[grade_mae]
    )
    print("Training Custom CNN...")
    cnn_history = custom_cnn.fit(
        train_ds,
        epochs=EPOCHS,
        validation_data=val_ds,
        callbacks=get_callbacks('custom_cnn'),
        verbose=1
    )
    cnn_loss = custom_cnn.evaluate(val_ds, verbose=0)
    print(f"\nCustom CNN — Val MSE: {cnn_loss[0]:.4f} | Grade MAE: {cnn_loss[1]:.3f} pts")
else:
    print("⚠️  No training data found. Add images to data/train/ and run again.")

In [ ]:
# Train MobileNetV2
if len(train_dataset) > 0:
    mobilenet_model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss='mse',
        metrics=[grade_mae]
    )
    print("Training MobileNetV2...")
    mobilenet_history = mobilenet_model.fit(
        train_ds,
        epochs=EPOCHS,
        validation_data=val_ds,
        callbacks=get_callbacks('mobilenet'),
        verbose=1
    )
    mobilenet_loss = mobilenet_model.evaluate(val_ds, verbose=0)
    print(f"\nMobileNetV2 — Val MSE: {mobilenet_loss[0]:.4f} | Grade MAE: {mobilenet_loss[1]:.3f} pts")

In [ ]:
# Train EfficientNetB3
if len(train_dataset) > 0:
    efficientnet_model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss='mse',
        metrics=[grade_mae]
    )
    print("Training EfficientNetB3...")
    efficientnet_history = efficientnet_model.fit(
        train_ds,
        epochs=EPOCHS,
        validation_data=val_ds,
        callbacks=get_callbacks('efficientnet'),
        verbose=1
    )
    efficientnet_loss = efficientnet_model.evaluate(val_ds, verbose=0)
    print(f"\nEfficientNetB3 — Val MSE: {efficientnet_loss[0]:.4f} | Grade MAE: {efficientnet_loss[1]:.3f} pts")

## Step 7: Evaluation & Grade Mapping

In [ ]:
# Training curve comparison (same plot style as plant project)
def plot_training_history(histories, names):
    colors = ['#003366', '#CC6600', '#006633']
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for history, name, color in zip(histories, names, colors):
        axes[0].plot(history.history['grade_mae'],     label=f'{name} Train', color=color, lw=2)
        axes[0].plot(history.history['val_grade_mae'], label=f'{name} Val',   color=color, lw=2, ls='--')
        axes[1].plot(history.history['loss'],          label=f'{name} Train', color=color, lw=2)
        axes[1].plot(history.history['val_loss'],      label=f'{name} Val',   color=color, lw=2, ls='--')
    axes[0].set_title('Grade MAE over Epochs (lower = better)', fontweight='bold')
    axes[0].set_ylabel('Mean Abs Error (grade points)')
    axes[0].legend(fontsize=7); axes[0].grid(alpha=0.3)
    axes[1].set_title('MSE Loss over Epochs', fontweight='bold')
    axes[1].set_ylabel('MSE')
    axes[1].legend(fontsize=7); axes[1].grid(alpha=0.3)
    for ax in axes:
        ax.set_xlabel('Epoch')
    plt.tight_layout()
    plt.show()

# Only plot if training ran
try:
    plot_training_history(
        [cnn_history, mobilenet_history, efficientnet_history],
        ['Custom CNN', 'MobileNetV2', 'EfficientNetB3']
    )
except NameError:
    print("Train the models first (Step 6) before plotting.")

In [ ]:
# Grade mapping functions — converts model output to grading scale
# Import from the standalone module
sys.path.append('../model')
from grade_mapping import predict_grade, GradeScales

# Test inference on a single card image
def grade_card(model, image_path, verbose=True):
    """
    Run grade prediction on a single card image.
    Returns predicted sub-scores and grade for each scale.
    """
    img  = load_img(image_path, target_size=IMG_SIZE)
    arr  = img_to_array(img) / 255.0
    arr  = np.expand_dims(arr, axis=0)

    pred = model.predict(arr, verbose=0)[0]  # Shape: (4,)
    sub_scores = {
        'centering': float(pred[0] * 10),
        'corners':   float(pred[1] * 10),
        'edges':     float(pred[2] * 10),
        'surface':   float(pred[3] * 10)
    }

    composite = sum(sub_scores[k] * GRADE_WEIGHTS[k] for k in GRADE_WEIGHTS)
    grades    = predict_grade(sub_scores)

    if verbose:
        fig, axes = plt.subplots(1, 2, figsize=(10, 5))
        axes[0].imshow(img_to_array(img).astype('uint8'))
        axes[0].set_title('Input Card', fontweight='bold')
        axes[0].axis('off')

        dim_scores = list(sub_scores.values())
        colors     = ['#2196F3', '#4CAF50', '#FF9800', '#F44336']
        bars = axes[1].barh(list(sub_scores.keys()), dim_scores, color=colors)
        axes[1].set_xlim(0, 10)
        axes[1].set_xlabel('Score (0-10)')
        axes[1].set_title(f'Sub-Scores\nComposite: {composite:.2f}', fontweight='bold')
        for bar, score in zip(bars, dim_scores):
            axes[1].text(score + 0.1, bar.get_y() + bar.get_height()/2,
                        f'{score:.1f}', va='center', fontweight='bold')
        plt.tight_layout()
        plt.show()

        print("\n=== Grade Predictions ===")
        for scale, grade in grades.items():
            print(f"  {scale:<10}: {grade}")

    return sub_scores, grades

print("Grade prediction function ready.")
print("Usage: sub_scores, grades = grade_card(best_model, 'path/to/card.jpg')")

## Step 8: ONNX Export (for GitHub Pages deployment)

In [ ]:
import tf2onnx
import onnx

def export_to_onnx(model, output_path, img_size=IMG_SIZE):
    """
    Export trained Keras model to ONNX format.
    ONNX models can run in the browser via ONNX Runtime Web — no server needed!
    This enables free GitHub Pages deployment.
    """
    input_signature = [tf.TensorSpec(
        shape=[None, img_size[0], img_size[1], 3],
        dtype=tf.float32,
        name='card_image'
    )]

    model_proto, _ = tf2onnx.convert.from_keras(
        model,
        input_signature=input_signature,
        opset=13,
        output_path=output_path
    )

    # Verify the exported model
    onnx_model = onnx.load(output_path)
    onnx.checker.check_model(onnx_model)

    size_mb = os.path.getsize(output_path) / (1024 * 1024)
    print(f"✅ ONNX model exported: {output_path}")
    print(f"   File size: {size_mb:.1f} MB")
    print(f"   This model can run in-browser via ONNX Runtime Web (free GitHub Pages hosting)")
    return output_path

# Export best model — update model name as needed
try:
    onnx_path = export_to_onnx(
        mobilenet_model,  # swap for whichever model performed best
        '../docs/card_grader.onnx'
    )
except NameError:
    print("Train a model first, then run this cell to export.")

---
## Summary — Full Pipeline

| Step | Technique | Purpose |
|------|-----------|----------|
| 1 | Configuration | IMG_SIZE=224, GRADE_WEIGHTS defined |
| 2 | eBay scraper / manual dataset | Collect labeled card images |
| 3 | CardGradingDataset class | Load image + JSON label pairs |
| 4a | PIL verify + load | Remove corrupt images |
| 4b | MD5 hash deduplication | Remove duplicate images |
| 4c | PCA + Isolation Forest | Remove visual outliers |
| 4d | Keras augmentation layer (no flip!) | Normalize & augment |
| 5 | Custom CNN, MobileNetV2, EfficientNetB3 | Three model architectures |
| 6 | MSE loss, grade_mae metric, callbacks | Training with regression output |
| 7 | Sub-score visualization + grade mapping | Evaluation across all scales |
| 8 | tf2onnx export | Browser-compatible model for GitHub Pages |